<div class="alert alert-block alert-info">
<b>Number of points for this notebook:</b> 2
<br>
<b>Deadline:</b> October 21, 2025 (Tuesday) 23:59 (end of day)
</div>

# Exercise 1.1. Train a multilayer perceptron (MLP) with pytorch.

The goal of this exercise is to get familiar with the basics of PyTorch and train a multilayer perceptron (MLP) model.

If you are not familiar with PyTorch, there is a number of good tutorials [here](https://pytorch.org/tutorials/index.html). We recommend the following ones:
* [What is PyTorch?](https://pytorch.org/tutorials/beginner/blitz/tensor_tutorial.html#sphx-glr-beginner-blitz-tensor-tutorial-py)
* [Autograd: Automatic Differentiation](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html#sphx-glr-beginner-blitz-autograd-tutorial-py)
* [Learning PyTorch with Examples](https://pytorch.org/tutorials/beginner/pytorch_with_examples.html)
* [Neural Networks](https://pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html#sphx-glr-beginner-blitz-neural-networks-tutorial-py)

In [2]:
skip_training = False  # Set this flag to True before validation and submission

In [ ]:
# During evaluation, this cell sets skip_training to True
# skip_training = True

In [2]:
pip install torch

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached fsspec-2025.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/109.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/109.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/109.3 MB 487.6 kB/s eta 0:03:45
   ---------------------------------------- 0.1/109.3 MB 744.7 kB/s eta 0:02:27
   ---------------------------------------- 0.1/109.3 MB 774.0 kB/s eta 0:02:22
   ---------------------------------------- 0.2/109.3 MB 1.1 MB/s eta 0:01:39
   ---------------------------------------- 0.5/109.3 MB 1.7 MB/s eta 0:01:03
   ---------------------------------------- 0.6/109.3 MB 1.8 MB/s eta 0:01:00
   ---------------------------------------- 1.0/109.3 MB 2.9 MB/s eta 0:00:38
   --------------------------------------


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\guzma\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F

import tools

In [3]:
# Select device which you are going to use for training
#device = torch.device("cuda:0")
device = torch.device("cpu")

In [4]:
if skip_training:
    # The models are always evaluated on CPU
    device = torch.device("cpu")

# Data

We will train the MLP on a toy regression problem.

In [34]:
# Let us generate toy data
def get_data():
    np.random.seed(2)
    x = np.random.randn(100, 1)
    x = np.sort(x, axis=0)

    targets = np.sin(x * 2 * np.pi / 3)
    targets = targets + 0.2 * np.random.randn(*targets.shape)

    # Convert to PyTorch tensors
    x = torch.FloatTensor(x)
    targets = torch.FloatTensor(targets)
    
    return x, targets

x, targets = get_data()
print(x.shape)
print(targets.shape)
# Plot the data
# fig, ax = plt.subplots(1)
# ax.plot(x, targets, '.')

torch.Size([100, 1])
torch.Size([100, 1])


# Multilayer perceptron (MLP) network with two hidden layers

We will create a simple multilayer perceptron (MLP) network. The model has
- input dimensionality 1
- one hidden layer with 10 units with Tanh nonlinearity
- one hidden layer with 11 units with Tanh nonlinearity
- linear output layer with output dimensionality 1 and no nonlinearity.

Hints:
* You may want to look at [this tutorial](https://pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html#sphx-glr-beginner-blitz-neural-networks-tutorial-py) for reference.
* You can use [`nn.Linear`](https://pytorch.org/docs/stable/nn.html?highlight=nn%20linear#torch.nn.Linear)
module to define the fully-connected layers of the MLP.
* Simple architectures are usually created using module [`torch.nn.Sequential`](https://pytorch.org/docs/stable/nn.html#torch.nn.Sequential). You do not have to use this module in this exercise.

Using Normal Linear Layers

In [35]:
class MLP(nn.Module):
    def __init__(self, n_inputs=1):
        super().__init__()
        self.fc1 = nn.Linear(n_inputs,10,bias=True,device=device)
        self.fc2 = nn.Linear(10,11,bias=True,device=device)
        self.final = nn.Linear(11,1,device=device)
    def forward(self, x):
        """
        Args:
          x of shape (n_samples, n_inputs): Model inputs.
        
        Returns:
          y of shape (n_samples, 1): Model outputs.
        """
        # YOUR CODE HERE
        #If x has a shape of (N,C,H,W) we will have to flatten it. 
        c1 = torch.tanh(self.fc1(x))
        c2 = torch.tanh(self.fc2(c1))
        output = torch.sigmoid(self.final(c2)) #prob between 0 and 1. 
        return output

Using Sequential Layers

In [36]:
# Let us create the network and make sure it can process a random input of the right shape
def test_MLP_shapes():
    n_inputs = 1
    n_samples = 10
    net = MLP()
    y = net(torch.randn(n_samples, n_inputs))
    assert y.shape == torch.Size([n_samples, 1]), f"Bad y.shape={y.shape}"
    print('Success')

test_MLP_shapes()

Success


## Train the MLP network

### Training loop

Your task is to implement the training loop.
You may find it useful to look at [this tutorial](https://pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html#sphx-glr-beginner-blitz-neural-networks-tutorial-py).
You should perform the following steps:
* Set all gradient values to zero.
* Calculate the output of the model for all training examples.
* Calculate the MSE loss (see [`mse_loss`](https://pytorch.org/docs/stable/nn.functional.html?highlight=mse_loss#torch.nn.functional.mse_loss)).
* Backpropagate the gradients: compute the gradients of the loss wrt to all the parameters of the model.
* Update the parameters of the model using the chosen optimizer.


Recommended hyperparameters:
* [Adam optimizer](https://pytorch.org/docs/stable/optim.html#torch.optim.Adam) with learning rate 0.01.
* You can process the data in the full-batch model (computing the gradients using all training data).
* Number of iterations (parameter updates): 200.

<span style="display:none">
Tip: In the training loop, add a comment saying "zero the gradient buffers" after optimizer.zero_grad().
</span>

In [37]:
# Create the model
model = MLP()
model.to(device)

MLP(
  (fc1): Linear(in_features=1, out_features=10, bias=True)
  (fc2): Linear(in_features=10, out_features=11, bias=True)
  (final): Linear(in_features=11, out_features=1, bias=True)
)

In [38]:
# Implement the training loop here
skip_training = False
model.train()
num_epochs = 200
epoch = 0
if not skip_training:
   criterion = nn.MSELoss(reduction='mean')
   optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
   x = x.to(device)
   targets = targets.to(device).float()
   for epoch in range(num_epochs): 
      #forward pass: 
      outputs = model(x)
      loss = criterion(outputs,targets)


      #backward and optimize: 
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      print(f'Epoch {epoch} and the loss is: {loss.item()}')
      epoch += 1 

Epoch 0 and the loss is: 0.7667593955993652
Epoch 1 and the loss is: 0.7416166067123413
Epoch 2 and the loss is: 0.7166061997413635
Epoch 3 and the loss is: 0.6921815276145935
Epoch 4 and the loss is: 0.6687784790992737
Epoch 5 and the loss is: 0.6469100713729858
Epoch 6 and the loss is: 0.6270782947540283
Epoch 7 and the loss is: 0.6096839904785156
Epoch 8 and the loss is: 0.5949652791023254
Epoch 9 and the loss is: 0.5829683542251587
Epoch 10 and the loss is: 0.5735538005828857
Epoch 11 and the loss is: 0.5664324760437012
Epoch 12 and the loss is: 0.5612229108810425
Epoch 13 and the loss is: 0.5575146675109863
Epoch 14 and the loss is: 0.5549204349517822
Epoch 15 and the loss is: 0.5531054735183716
Epoch 16 and the loss is: 0.5517979264259338
Epoch 17 and the loss is: 0.5507873296737671
Epoch 18 and the loss is: 0.5499165654182434
Epoch 19 and the loss is: 0.5490723848342896
Epoch 20 and the loss is: 0.5481753945350647
Epoch 21 and the loss is: 0.5471718311309814
Epoch 22 and the los

In [39]:
# Save the model to disk (the pth-files have to be submitted together with your notebook)
# Set confirm=False if you do not want to be asked for confirmation before saving.
if not skip_training:
    tools.save_model(model, '2_mlp.pth', confirm=True)

Model saved to 2_mlp.pth.


In [40]:
skip_training = True 
if skip_training:
    model = MLP()
    tools.load_model(model, '2_mlp.pth', device)

Model loaded from 2_mlp.pth.


c:\Users\guzma\4 IMAT\PRIMER CUATRI\Machines Learning and Perception\Práctica 2\tools.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(filename, map_location=

In [ ]:
# Plot the trained MLP
with torch.no_grad():
    fig, ax = plt.subplots(1)
    ax.plot(x, targets, '.')
    y = model(x)
    ax.plot(x, y.numpy(), 'r-')
    ax.grid(True)

: 

In [ ]:
# This cell tests MLP
model.eval()
with torch.no_grad(): 
    x_dev = x.to(device)
    targets_dev = targets.to(device).long().view(-1)
    logits = model(x_dev)
    preds = (logits >= 0.5).long().view(-1)
    acc = (preds == targets_dev).float().mean().item()

    print(f'Accuracy of the network is: {acc}')    

Accuracy of the network is: 0.699999988079071
